<a href="https://colab.research.google.com/github/Sruteeshsatya/meeting-assistant/blob/main/colab_launch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎙️ AI Meeting Assistant — Colab launcher

Runs the full app (faster-whisper → Gemini refinement → Gemini documentation) on a Colab GPU and gives you a public link to the Streamlit interface.

**Before you start**
1. *Runtime → Change runtime type → T4 GPU*.
2. Add your Gemini key as a Colab secret: click the 🔑 icon in the left sidebar → *Add new secret* → name `GEMINI_API_KEY`, paste the key, and enable *Notebook access*.
3. Run the cells in order.

In [1]:
# 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU - the app will run on CPU with a smaller Whisper model"

name, memory.total [MiB]
Tesla T4, 15360 MiB


## 2. Get the code
Either set `REPO_URL` to your GitHub repository, or leave it empty and upload `meeting-assistant.zip` when prompted.

In [2]:
import os, zipfile, glob

REPO_URL = ""  # e.g. "https://github.com/<you>/meeting-assistant.git"

os.chdir("/content")
if REPO_URL:
    !rm -rf meeting-assistant && git clone -q {REPO_URL} meeting-assistant
elif not os.path.exists("/content/meeting-assistant/app.py"):
    from google.colab import files
    uploaded = files.upload()  # choose meeting-assistant.zip
    zip_name = next(iter(uploaded))
    with zipfile.ZipFile(zip_name) as zf:
        zf.extractall("/content")
app = glob.glob("/content/**/app.py", recursive=True)[0]
PROJECT = os.path.dirname(app)
os.chdir(PROJECT)
print("Project folder:", PROJECT)

Saving meeting-assistant-2.zip to meeting-assistant-2.zip
Project folder: /content/meeting-assistant


In [3]:
# 3. Install dependencies (ffmpeg is already on Colab)
!pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 117.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 72.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 105.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 94.9 MB/s eta 0:00:00


In [4]:
# 4. Load the Gemini key from Colab secrets
import os
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
# Optional overrides:
os.environ["REFINE_MODEL"] = "gemini-3.5-flash-lite"
os.environ["DOCS_MODEL"] = "gemini-3.5-flash-lite"
# os.environ["WHISPER_MODEL"] = "large-v3"   # 'auto' = large-v3 on GPU
print("Key loaded:", bool(os.environ["GEMINI_API_KEY"]))

Key loaded: True


In [5]:
import os, sys, glob
PROJECT = os.path.dirname(os.path.dirname(glob.glob('/content/**/meeting_assistant/__init__.py', recursive=True)[0]))
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
print("Ready:", PROJECT)

Ready: /content/meeting-assistant


In [6]:
# 5. (Recommended) Pre-download the Whisper model so the first run in the app is fast (~3 GB for large-v3)
from meeting_assistant.transcribe import WhisperTranscriber
stt = WhisperTranscriber(os.environ.get("WHISPER_MODEL", "auto"))
print("Loaded:", stt.name)
del stt

Loaded: faster-whisper large-v3 (cuda)


In [7]:
# 6. Run the tests (offline, uses fake models) - should all pass
!python -m pytest -q

.................................                                        [100%]
33 passed in 4.97s


In [8]:
# 7. Start the app and open a public tunnel
import subprocess, time, re, os

subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
                 stdout=open("streamlit.log", "w"), stderr=subprocess.STDOUT, env=os.environ.copy())

if not os.path.exists("/usr/local/bin/cloudflared"):
    !wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /usr/local/bin/cloudflared
tunnel = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8501"],
                          stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    time.sleep(1)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("tunnel.log").read())
    if m:
        url = m.group(0); break
print("Open the app here:", url or "tunnel not ready - check tunnel.log")

Open the app here: https://rainbow-instant-battle-borough.trycloudflare.com


## Optional: command-line run
Processes one file end to end and saves every output under `runs/` (useful for the sample-output deliverable).

In [ ]:
from google.colab import files
up = files.upload()
audio = next(iter(up))
!python cli.py "{audio}" --context "add domain terms and names here"
!ls -la runs/*